In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio
import torch

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [ ]:
ds = load_dataset("Isma/librispeech_tiny")
ds = ds["10mn"]

def prepare_batch(batch):
    print(batch["audio"]["array"])
    waveform = batch["audio"]["array"]
    sr = batch["audio"]["sampling_rate"]
    batch["input_features"] = processor(
        waveform,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features[0]
    return batch

dataset = ds.map(prepare_batch)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

[0.02813721 0.02984619 0.01861572 ... 0.03112793 0.03240967 0.02627563]
[-3.0517578e-05  6.1035156e-05  9.1552734e-05 ...  0.0000000e+00
  0.0000000e+00  3.0517578e-05]
[2.1362305e-04 1.8310547e-04 1.8310547e-04 ... 9.1552734e-05 1.2207031e-04
 1.2207031e-04]
[-0.00408936 -0.00192261 -0.00094604 ... -0.01248169 -0.01254272
 -0.0083313 ]
[ 2.4414062e-04 -3.0517578e-05 -5.1879883e-04 ... -1.6784668e-03
 -3.2348633e-03 -2.6855469e-03]
[-1.0681152e-03 -5.1879883e-04  3.0517578e-05 ...  2.2277832e-03
  2.1972656e-03  2.4108887e-03]
[ 0.00405884  0.0020752   0.00335693 ...  0.00128174 -0.00027466
 -0.00164795]
[-1.2817383e-03 -7.3242188e-04  3.3569336e-04 ... -9.7656250e-04
  6.1035156e-05  3.6621094e-04]
[0. 0. 0. ... 0. 0. 0.]
[ 0.00015259 -0.00067139 -0.00042725 ... -0.0017395   0.0057373
  0.00753784]
[ 1.2207031e-04 -6.1035156e-05 -4.8828125e-04 ... -2.3498535e-03
 -2.5634766e-03 -3.2348633e-03]
[-0.00024414  0.00119019  0.00064087 ...  0.00079346  0.00064087
  0.00036621]
[ 0.00112915 

In [ ]:
transcriptions = []
for sample in dataset:
    input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to(device)

    generated_ids = model.generate(input_features, language="en", task="translate")
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    transcriptions.append(text)

for i, t in enumerate(transcriptions):
    print(f"Sample {i}: {t}")

Sample 0:  passed so close to D'Artagnan's face
Sample 1:  They were therefore resolved, if possible, to excite him to some violent passion.
Sample 2:  But in all she did for him, she felt like the executioner who gives restoratives to the wretch that has fainted on the rack or the wheel. What right had she, she thought, to multiply to him his moments of torture?
Sample 3:  I sank deep, deep down until at last I got to the bottom.
Sample 4:  and ungenial when the sleeping wind has awoke in the east or when the done clouds thickly veil the sky
Sample 5:  I have no pleasure in talking to undutiful children.
Sample 6:  To whom it was said he had sold his sister, Miss Churchill. Ballingbrook was in his meridian, and Richelieu in his dawn. Gallantry found its convenience in a certain medley of ranks. Men were equalized by the same vices as they were later on, perhaps by the same ideas.
Sample 7:  Yet no sooner did a daring rebel or murderer gather a band of robbers around him and begin to k